In [1]:
import pandas as pd
import json
import csv
from datetime import datetime

# Configurações de exibição do Pandas para facilitar o debug
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [2]:
import urllib.request
import tarfile
import os

# 1. URL corrigida com 'id_' no final do timestamp (padrão do Internet Archive para arquivo cru)
url = "https://web.archive.org/web/20210420235203id_/http://research.moodle.org/158/2/export.tar.gz"
arquivo_compactado = "export.tar.gz"
arquivo_verificacao = "mdl_logstore_standard_log.csv" 

def baixar_e_extrair_dados():
    if os.path.exists(arquivo_verificacao):
        print("✅ Os dados brutos já estão presentes na pasta. Pulando o download!")
        return

    print(f"📥 Iniciando o download do dataset gigante... (Isso pode demorar dependendo da conexão)")
    try:
        # 2. Criando um disfarce (User-Agent) para não sermos bloqueados
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'})
        
        # 3. Baixando o arquivo em "pedacinhos" para não travar a memória do computador
        with urllib.request.urlopen(req) as response, open(arquivo_compactado, 'wb') as out_file:
            while chunk := response.read(8192):
                out_file.write(chunk)
                
        print("📦 Download real concluído! Iniciando a extração dos arquivos...")

        # Extrai o .tar.gz
        with tarfile.open(arquivo_compactado, "r:gz") as tar:
            tar.extractall(path=".")
        
        print("🧹 Extração finalizada! Deletando o arquivo compactado para liberar espaço...")
        os.remove(arquivo_compactado)
        
        print("🚀 Tudo pronto! O dataset está descompactado e pronto para o Pandas.")
        
    except Exception as e:
        print(f"❌ Ocorreu um erro durante o processo: {e}")
        # Limpa o arquivo corrompido para não atrapalhar a próxima tentativa
        if os.path.exists(arquivo_compactado):
            os.remove(arquivo_compactado)

# Executa a função
baixar_e_extrair_dados()

📥 Iniciando o download do dataset gigante... (Isso pode demorar dependendo da conexão)
📦 Download real concluído! Iniciando a extração dos arquivos...


C:\Users\Danilus04\AppData\Local\Temp\ipykernel_30008\3219628129.py:29: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=".")


🧹 Extração finalizada! Deletando o arquivo compactado para liberar espaço...
🚀 Tudo pronto! O dataset está descompactado e pronto para o Pandas.


In [3]:
# Mapeamento oficial: Evento do Moodle -> Verbo xAPI
MOODLE_TO_XAPI_VERBS = {
    r'\core\event\course_viewed': 'http://id.tincanapi.com/verb/viewed',
    r'\mod_quiz\event\attempt_started': 'http://adlnet.gov/expapi/verbs/launched',
    r'\mod_quiz\event\attempt_submitted': 'http://activitystrea.ms/schema/1.0/submit',
    r'\mod_quiz\event\attempt_reviewed': 'http://id.tincanapi.com/verb/reviewed',
    r'\mod_assign\event\assessable_submitted': 'http://activitystrea.ms/schema/1.0/submit',
    r'\core\event\course_completed': 'http://adlnet.gov/expapi/verbs/completed',
    r'\core\event\badge_awarded': 'http://adlnet.gov/expapi/verbs/earned',
    r'\core\event\user_loggedin': 'https://w3id.org/xapi/adl/verbs/logged-in',
}

def obter_verbo_xapi(eventname):
    # Fallback para 'interacted' se o evento não estiver mapeado
    return MOODLE_TO_XAPI_VERBS.get(eventname, 'http://adlnet.gov/expapi/verbs/interacted')

In [4]:
def formatar_timestamp(unix_timestamp):
    try:
        return datetime.fromtimestamp(int(unix_timestamp)).isoformat() + "Z"
    except:
        return datetime.utcnow().isoformat() + "Z"

def construir_object_id(component, instance_id, course_id):
    nome_modulo = component.replace("mod_", "") if component.startswith("mod_") else component
    return f"http://seumoodle.com/mod/{nome_modulo}/view.php?id={instance_id}&course={course_id}"

In [5]:
def transformar_logs_para_xapi(df_logs):
    statements = []
    for _, row in df_logs.iterrows():
        userid = str(row.get('username', ''))
        if userid in ['', '0', 'nan', '-1']: continue
            
        statement = {
            "actor": {"objectType": "Agent", "account": {"homePage": "http://seumoodle.com", "name": userid}},
            "verb": {
                "id": obter_verbo_xapi(row.get('eventname', '')),
                "display": {"en-US": row.get('action', 'interacted')}
            },
            "object": {
                "objectType": "Activity",
                "id": construir_object_id(row.get('component', ''), row.get('contextinstanceid', '0'), row.get('courseid', '0')),
                "definition": {"name": {"en-US": f"Atividade: {row.get('component', 'core')}"}}
            },
            "timestamp": formatar_timestamp(row.get('timecreated', 0)),
            "context": {
                "extensions": {
                    "http://moodle.org/ext/eventname": row.get('eventname', ''),
                    "http://moodle.org/ext/component": row.get('component', '')
                }
            }
        }
        statements.append(statement)
    return statements

def transformar_notas_para_xapi(df_notas):
    statements = []
    # Filtra apenas linhas que possuem nota e nota máxima
    df_validas = df_notas.dropna(subset=['rawgrade', 'rawgrademax'])
    
    for _, row in df_validas.iterrows():
        userid = str(row.get('username', ''))
        if userid in ['', '0', 'nan', '-1']: continue

        item_id = str(row.get('itemid', '0'))
        
        statement = {
            "actor": {"objectType": "Agent", "account": {"name": userid}},
            "verb": {
                "id": "http://adlnet.gov/expapi/verbs/scored",
                "display": {"en-US": "scored"}
            },
            "object": {
                "objectType": "Activity",
                "id": f"http://seumoodle.com/grade/item/{item_id}"
            },
            "result": {
                "score": {
                    "raw": float(row['rawgrade']),
                    "max": float(row['rawgrademax'])
                }
            },
            "timestamp": formatar_timestamp(row.get('timemodified', 0)),
            "context": {
                "contextActivities": {
                    "parent": [
                        {
                            "id": f"http://seumoodle.com/mod/quiz/view.php?id={item_id}",
                            "definition": {
                                "type": "activitytype/course",
                                "name": {"en": f"Curso/Matéria ID {row.get('courseid', item_id)}"}
                            }
                        }
                    ]
                }
            }
        }
        statements.append(statement)
    return statements

In [7]:
# 1. Carregamento dos Datasets
print("📖 Carregando arquivos originais...")
df_logs = pd.read_csv("export/mdl_logstore_standard_log.csv", low_memory=False)
df_notas = pd.read_csv("export/mdl_grade_grades_history.csv", low_memory=False)

# 2. Transformação
print(f"⚙️ Processando {len(df_logs)} logs de eventos...")
statements_logs = transformar_logs_para_xapi(df_logs)

print(f"⚙️ Processando {len(df_notas)} registros de notas...")
statements_notas = transformar_notas_para_xapi(df_notas)

# 3. União dos Statements
# Juntamos as duas listas em uma só
statements_totais = statements_logs + statements_notas

# 4. Exportação
caminho_output = "xapi_statements_completos.json"
print(f"💾 Salvando {len(statements_totais)} statements unificados em '{caminho_output}'...")

with open(caminho_output, "w", encoding="utf-8") as f:
    json.dump(statements_totais, f, indent=4, ensure_ascii=False)

print("🚀 Processo concluído! O arquivo está pronto para a sua métrica de pontuação.")

📖 Carregando arquivos originais...
⚙️ Processando 2391762 logs de eventos...
⚙️ Processando 70037 registros de notas...
💾 Salvando 2400482 statements unificados em 'xapi_statements_completos.json'...
🚀 Processo concluído! O arquivo está pronto para a sua métrica de pontuação.
